In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [5]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [6]:
# Create a state

class LLMState(TypedDict):
    question: str
    answer: str

In [7]:
def llm_qa(state: LLMState) -> LLMState:
    #Extract the question from state
    question = state["question"]

    #Form a prompt for the LLM
    prompt = f"Answer the following question: {question}"

    #Ask that question to the LLM
    answer = model.invoke(prompt).content

    #Return the answer in a new state
    state["answer"] = answer

    return state

In [8]:
# Create our graph
graph = StateGraph(LLMState)

# add node (pass the function, not its result)
graph.add_node("llm_qa", llm_qa)

# add edges
graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)

# compile the graph
workflow = graph.compile()

In [9]:
#execute the workflow
initial_state = LLMState(question="What is the capital of France?")
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'What is the capital of France?', 'answer': 'The capital of France is Paris.'}


In [12]:
model.invoke("How far is moon from Earth?").content      #We could have directly used this but we did it in the previous way just to show how to use langgraph to create a workflow with LLMs.

"The average distance from the Earth to the Moon is approximately 384,400 kilometers (238,900 miles). This distance is constantly changing due to the elliptical shape of the Moon's orbit around the Earth. At its closest point (called perigee), the Moon is about 356,400 kilometers (221,500 miles) away, and at its farthest point (apogee), it is about 405,500 kilometers (252,000 miles) away."